# 03_demotion_timeline

A structured tracker for the observational papers on lunar mini-magnetospheres,
built to support two figures in the literature review presentation:

- **Slide 3.1**, the demotion timeline: how confidence in the mini-magnetosphere
  framing rose and fell as instrumentation improved.
- **Slide 3.2**, "shielding is always partial": the measured exclusion fractions.

The intended figure is a dumbbell chart — one row per paper, two markers per row
(claimed structure scale, and the proton gyroradius at that paper's own field),
log length axis. Every pair points the same way: the gyroradius sits to the
right of the claimed structure, in every paper.

## The argument the data serves

Crustal anomalies are obstacles smaller than the scale on which solar wind ions
can respond. Electrons are magnetised; protons are not. Everything observed —
partial shielding, shock-like discontinuities, reflected protons,
electron-only reconnection — follows from that inequality.
The timeline is not monotonic. Claim strength peaks at Kurata 2005 (declarative
title, two magnetometer passes, no plasma instrument) and declines thereafter as
plasma and ion measurements arrive.

## Why the columns are shaped this way

Several columns exist to prevent the figure from lying:

- **`B_peak_nT` vs `B_paper_nT`** — the strongest field the paper reports at its
  own observation, versus the field it feeds into its own scale argument. These
  differ, and the difference is often the finding (Lin quotes 12 nT in his
  coherence criterion while observing 30 nT).
- **`L_basis` and `L_orientation`** — "structure scale" does not mean the same
  thing twice. Kurata's 30 km is a *vertical* lower bound set by spacecraft
  periapsis; Wieser's 360 km is a *horizontal* surface footprint imaged from
  200 km. Plotting both on one length axis without flagging this is the most
  attackable thing the figure could do.
- **`v_kms` and `v_basis`** — gyroradii are computed at each paper's own
  measured solar wind speed where quoted, nominal 400 km/s otherwise. Using a
  single speed throughout would break the like-for-like comparison; Halekas's
  event was 303 km/s.
- **`exclusion_pct` + `exclusion_species` + `exclusion_basis`** — the depletion
  numbers are not commensurable. Halekas is a normalised electron density drop;
  Wieser is an H-ENA flux reduction measured against an already-enhanced ring.
  The basis string carries more information than the number.
- **`alt_context`** — spacecraft altitude, structure altitude and the altitude
  at which a field value was modelled are three different things, and Wieser
  has all three in one row.

## Convention

Proton gyroradii use the **convective** speed, not the thermal speed. At the
nose, `B_n = 0` and the flow is normal, so the bulk drift projects fully onto
the perpendicular direction — this is geometry, not a comparison of magnitudes.
Protons complete a fraction of one orbit per crossing, so there is no gyrophase
averaging to justify a thermal value. Electrons, which complete hundreds, are a
different case.

`r_p_therm_km` is computed anyway, because the divergence between the two is
itself evidence: it runs a factor of ~9 under nominal conditions and ~21 at the
Halekas event, and the thermal substitution is what lets Lin's ~30 km criterion
look reasonable.

In [4]:
import numpy as np
import pandas as pd

# --- constants -------------------------------------------------------------
M_P    = 1.67262192e-27   # kg
Q_E    = 1.602176634e-19  # C
V_SW   = 400e3            # m/s, nominal convective (bulk) speed
T_P_EV = 10.0             # eV, nominal proton temperature
V_TH   = np.sqrt(2 * T_P_EV * Q_E / M_P)   # m/s, ~43.8 km/s
EPS0 = 8.8541878128e-12
C_L  = 2.99792458e8

def r_p(B_nT, v):
    """Proton gyroradius in km. B in nT, v in m/s."""
    if B_nT is None or np.isnan(B_nT):
        return np.nan
    return (M_P * v) / (Q_E * B_nT * 1e-9) / 1e3


def d_i(n_cm3):
    """Ion inertial length in km. n in cm^-3."""
    if n_cm3 is None or np.isnan(n_cm3):
        return np.nan
    n = n_cm3 * 1e6
    w_pi = np.sqrt(n * Q_E**2 / (EPS0 * M_P))
    return C_L / w_pi / 1e3

# --- schema ----------------------------------------------------------------
# B_peak_nT : strongest field the paper reports at its own observation.
#             Use THIS for r_p, so the comparison is like-for-like.
# B_paper_nT: the field the paper plugs into its own scale argument, if any.
#             Differs from B_peak - that difference is often the finding.
# L_basis   : how the scale was arrived at. Will not mean the same thing twice.

COLS = ["paper", "year", "alt_min_km", "alt_max_km", "alt_context",
        "B_peak_nT", "B_peak_context", "B_paper_nT", "B_paper_context",
        "L_min_km", "L_max_km", "L_basis", "L_orientation",
        "v_kms", "v_basis", "n_cm3", "T_p_eV", "B_imf_nT", "P_sw_nPa",
        "instrument_class", "ions_measured", "N_evidence",
        "name_used", "exclusion_pct", "exclusion_species", "exclusion_basis"]

rows = [
    dict(paper="Lin", year=1998,
         alt_min_km=100.0, alt_max_km=100.0,
         alt_context="spacecraft",
         B_peak_nT=30.0,
         B_peak_context="peak |B| in shock enhancement, Fig 3; contours at 20 and 27 nT",
         B_paper_nT=12.0,
         B_paper_context="field in his coherence criterion - source of 12 nT never stated",
         L_min_km=100.0, L_max_km=500.0,
         L_basis="abstract assertion, 'across'; inherited from crustal patch size, "
                 "not a measured boundary. NOT the 100 km altitude.",
         L_orientation="horizontal",
         v_kms=400.0, v_basis="nominal - no local SW velocity quoted",
         n_cm3=np.nan, T_p_eV=np.nan, B_imf_nT=10.0, P_sw_nPa=3.0,
         instrument_class="field+electrons",
         ions_measured=False,
         N_evidence="4 of 5 consecutive orbits",
         name_used="miniature magnetosphere / limb shock vs limb compression (hedged in text)",
         exclusion_pct=np.nan, exclusion_species=None,
         exclusion_basis="not measured; shock inferred from electron energization + 2.5 Hz whistlers"),

    dict(paper="Kurata", year=2005,
         alt_min_km=26.0, alt_max_km=29.1,
         alt_context="spacecraft",
         B_peak_nT=35.0,
         B_peak_context="peak observed total intensity, day 68",
         B_paper_nT=np.nan,
         B_paper_context="no scale argument made",
         L_min_km=30.0, L_max_km=np.nan,
         L_basis="spacecraft altitude - lower bound, no boundary crossing",
         L_orientation="vertical",
         v_kms=400.0, v_basis="nominal - only P_sw quoted",
         n_cm3=np.nan, T_p_eV=np.nan, B_imf_nT=np.nan, P_sw_nPa=3.5,
         instrument_class="field",
         ions_measured=False,
         N_evidence="2 passes",
         name_used="mini-magnetosphere (declarative, in title)",
         exclusion_pct=np.nan, exclusion_species=None,
         exclusion_basis="not measured; no plasma instrument"),

    dict(paper="Halekas", year=2008,
         alt_min_km=30.0, alt_max_km=36.0,
         alt_context="spacecraft; 30 km at 2nd cavity (17:33), higher at 1st (15:43) - "
                     "abstract's '30 km' is not both orbits",
         B_peak_nT=18.0,
         B_peak_context="peak |B| in 2nd cavity, read off Fig 3 (not in text); 1st cavity ~15 nT",
         B_paper_nT=np.nan,
         B_paper_context="no field-based scale argument; scaling argued via d_i instead",
         L_min_km=np.nan, L_max_km=np.nan,
         L_basis="no structure size quoted. Cavity ~10 deg downstream of CA centre - "
                 "that is a displacement, not an extent.",
         L_orientation=None,
         v_kms=303.0, v_basis="Wind SWE, time-shifted; 96th pct (low)",
         n_cm3=15.8, T_p_eV=1.11, B_imf_nT=2.36, P_sw_nPa=np.nan,
         instrument_class="field+electrons",
         ions_measured=False,
         N_evidence="2 consecutive orbits in ~7 months; null argued from ~1000 wake voids",
         name_used="'what may be' inner region of a mini-magnetosphere (interrogative title)",
         exclusion_pct=95.0, exclusion_species="electron",
         exclusion_basis="n/n_0 bottoms at ~0.05 both cavities (Fig 3); 3-d kappa-fit moments "
                         "NORMALISED to SW average, relative not absolute"),

    dict(paper="Wieser", year=2010,
         alt_min_km=200.0, alt_max_km=200.0,
         alt_context="spacecraft. Structure imaged AT SURFACE; field quoted at 30 km. "
                     "Three different altitudes in one row - do not plot 200 km as "
                     "the interaction altitude.",
         B_peak_nT=np.nan,
         B_peak_context="no magnetometer in this analysis",
         B_paper_nT=20.0,
         B_paper_context="imported: LP-derived model value at 30 km alt "
                         "(Mitchell 2008; Richmond & Hood 2008). 100 nT at surface.",
         L_min_km=360.0, L_max_km=360.0,
         L_basis="ENA surface footprint diameter, 150-600 eV band; "
                 "+300 km thick enhanced-flux annulus outside it",
         L_orientation="horizontal",
         v_kms=333.0, v_basis="from measured mean SW proton energy 580 eV. "
                              "NB paper computes r_p using 1 keV, giving 100 km not 174 km.",
         n_cm3=np.nan, T_p_eV=np.nan, B_imf_nT=5.0, P_sw_nPa=1.25,
         instrument_class="ENA+ions",
         ions_measured=True,
         N_evidence="1 day (17 Jun 2009), consistent on each anomaly pass",
         name_used="mini-magnetosphere, declarative + 'direct proof'; "
                   "but 'partial void' in same abstract",
         exclusion_pct=50.0, exclusion_species="H ENA proxy for proton",
         exclusion_basis="ENA flux reduction 150-600 eV, relative to SURROUNDING ENHANCED RING "
                         "not undisturbed terrain - true depletion is smaller. "
                         "Depletion vanishes below 100 eV."),

    dict(paper="Lue",       year=2011),
    dict(paper="Saito",     year=2012),
    dict(paper="Vorburger", year=2012),
    dict(paper="Xie",       year=2021),
]

df = pd.DataFrame(rows).reindex(columns=COLS)

# --- derived ---------------------------------------------------------------
df["alt_mid_km"]   = df[["alt_min_km", "alt_max_km"]].mean(axis=1)
df["L_mid_km"]     = df[["L_min_km", "L_max_km"]].mean(axis=1)
df["v_ms"]         = df["v_kms"] * 1e3
df["r_p_conv_km"]  = df.apply(lambda r: r_p(r["B_peak_nT"], r["v_ms"]), axis=1)
df["r_p_paper_km"] = df.apply(lambda r: r_p(r["B_paper_nT"], r["v_ms"]), axis=1)
df["d_i_km"]       = df["n_cm3"].apply(d_i)
df["L_over_rp"]    = df["L_mid_km"] / df["r_p_conv_km"].fillna(df["r_p_paper_km"])

# self-checks
assert abs(r_p(12.0, 400e3) - 348) < 2, "Lin criterion field"
assert abs(r_p(35.0, 400e3) - 119) < 2, "Kurata peak field"
assert abs(r_p(18.0, 303e3) - 175) < 2, "Halekas cavity field"
assert abs(r_p(20.0, 333e3) - 174) < 2, "Wieser, at their own 580 eV not 1 keV"
assert abs(d_i(15.8)  - 57) < 1,        "Halekas d_i reproduces their 57 km"

df

,paper,year,alt_min_km,alt_max_km,alt_context,B_peak_nT,B_peak_context,B_paper_nT,B_paper_context,L_min_km,...,exclusion_pct,exclusion_species,exclusion_basis,alt_mid_km,L_mid_km,v_ms,r_p_conv_km,r_p_paper_km,d_i_km,L_over_rp
0,Lin,1998,100.0,100.0,spacecraft,30.0,"peak |B| in shock enhancement, Fig 3; contours...",12.0,field in his coherence criterion - source of 1...,100.0,...,NaN,NaN,not measured; shock inferred from electron ene...,100.00,300.0,400000.0,139.195799,347.989496,NaN,2.155237
1,Kurata,2005,26.0,29.1,spacecraft,35.0,"peak observed total intensity, day 68",NaN,no scale argument made,30.0,...,NaN,NaN,not measured; no plasma instrument,27.55,30.0,400000.0,119.310684,NaN,NaN,0.251444
2,Halekas,2008,30.0,36.0,"spacecraft; 30 km at 2nd cavity (17:33), highe...",18.0,"peak |B| in 2nd cavity, read off Fig 3 (not in...",NaN,no field-based scale argument; scaling argued ...,NaN,...,95.0,electron,n/n_0 bottoms at ~0.05 both cavities (Fig 3); ...,33.00,NaN,303000.0,175.734696,NaN,57.286861,NaN
3,Wieser,2010,200.0,200.0,spacecraft. Structure imaged AT SURFACE; field...,NaN,no magnetometer in this analysis,20.0,imported: LP-derived model value at 30 km alt ...,360.0,...,50.0,H ENA proxy for proton,"ENA flux reduction 150-600 eV, relative to SUR...",200.00,360.0,333000.0,NaN,173.820753,NaN,2.071099
4,Lue,2011,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Saito,2012,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Vorburger,2012,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Xie,2021,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
